<a href="https://colab.research.google.com/github/HereLiesAz/CueDetat/blob/main/ml/pocket_detector_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cue d'Etat — Pocket Detector Training

Trains a **YOLOv8n** model on a billiards dataset and exports **TFLite FP16**
ready to drop into the `PocketDetector` interface in the app.

**Runtime:** GPU — Runtime → Change runtime type → T4 GPU.

The dataset may include multiple classes (balls, table, pockets, etc.).
The model is trained on all classes as-is. You only need to know which
class index corresponds to pockets — set that in Section 2 and the
smoke-test and Android integration will use it correctly.

## 1. Install

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.7/112.6 GB disk)


In [ ]:
import zipfile
import os
import yaml

# List of zip files to extract
datasets_zips = [
    '/content/CueTor Billiards Training.v8i.yolov8.zip',
    '/content/Pool Table Detection.v6i.yolov8.zip'
]

def extract_and_inspect(zip_path):
    folder_name = os.path.basename(zip_path).replace('.zip', '').replace(' ', '_')
    extract_to = f'/content/{folder_name}'

    print(f'--- Processing: {os.path.basename(zip_path)} ---')
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)

        yaml_path = os.path.join(extract_to, 'data.yaml')
        if os.path.exists(yaml_path):
            with open(yaml_path, 'r') as f:
                data = yaml.safe_load(f)
                names = data.get('names', [])
                print(f'Location: {extract_to}')
                print(f'Classes: {names}')
        return extract_to
    except Exception as e:
        print(f'Error extracting {zip_path}: {e}')
        return None

# Extract zip files and keep track of folders
extracted_paths = []
for ds in datasets_zips:
    if os.path.exists(ds):
        path = extract_and_inspect(ds)
        if path: extracted_paths.append(path)
    else:
        print(f'File not found: {ds}')

# Add the Roboflow downloaded folder to the list
roboflow_dir = '/content/pool-balls-yolov11m-14'
if os.path.exists(roboflow_dir):
    extracted_paths.append(roboflow_dir)
    print(f'Added Roboflow dataset: {roboflow_dir}')

--- Processing: CueTor Billiards Training.v8i.yolov8.zip ---
Location: /content/CueTor_Billiards_Training.v8i.yolov8
Classes: ['0', '1', '10', '11', '12', '13', '14', '15', '2', '3', '4', '5', '6', '7', '8', '9']
--- Processing: Pool Table Detection.v6i.yolov8.zip ---
Location: /content/Pool_Table_Detection.v6i.yolov8
Classes: ['pool-table', 'pool-table-hole', 'pool-table-side']
Added Roboflow dataset: /content/pool-balls-yolov11m-14


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="sPwF0dxJDJId1E7SFQFp")
project = rf.workspace("wrckmn1").project("pool-balls-yolov11m")
version = project.version(14)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to pool-balls-yolov11m-14 in yolov8:: 100%|██████████| 92542/92542 [00:14<00:00, 6358.48it/s]


In [ ]:
import os
import shutil
import yaml

combined_dir = '/content/combined_dataset'
if os.path.exists(combined_dir): shutil.rmtree(combined_dir)
os.makedirs(combined_dir, exist_ok=True)

for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(combined_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(combined_dir, split, 'labels'), exist_ok=True)

master_classes = []

print("Merging datasets...")
for folder in extracted_paths:
    y_path = os.path.join(folder, 'data.yaml')
    if not os.path.exists(y_path): continue

    with open(y_path, 'r') as f:
        cfg = yaml.safe_load(f)
        names = cfg.get('names', [])
        # Standardize names to list
        if isinstance(names, dict):
            names = [names[i] for i in sorted(names.keys())]

        # Build master class list
        class_map = {}
        for i, name in enumerate(names):
            if name not in master_classes:
                master_classes.append(name)
            class_map[i] = master_classes.index(name)

    # Copy and re-index labels
    for split in ['train', 'valid', 'test']:
        src_img = os.path.join(folder, split, 'images')
        src_lbl = os.path.join(folder, split, 'labels')

        if os.path.exists(src_img):
            for f in os.listdir(src_img):
                shutil.copy2(os.path.join(src_img, f), os.path.join(combined_dir, split, 'images', f))

        if os.path.exists(src_lbl):
            for f in os.listdir(src_lbl):
                with open(os.path.join(src_lbl, f), 'r') as lf:
                    lines = lf.readlines()
                with open(os.path.join(combined_dir, split, 'labels', f), 'w') as lf:
                    for line in lines:
                        parts = line.split()
                        if parts:
                            old_idx = int(parts[0])
                            parts[0] = str(class_map.get(old_idx, old_idx))
                            lf.write(' '.join(parts) + '\n')

combined_yaml = {
    'train': f'{combined_dir}/train/images',
    'val': f'{combined_dir}/valid/images',
    'test': f'{combined_dir}/test/images',
    'nc': len(master_classes),
    'names': master_classes
}

with open(os.path.join(combined_dir, 'data.yaml'), 'w') as f:
    yaml.dump(combined_yaml, f)

print(f"Done! Combined dataset at {combined_dir}")
print(f"Combined Classes: {master_classes}")

Merging datasets...
Done! Combined dataset at /content/combined_dataset
Combined Classes: ['0', '1', '10', '11', '12', '13', '14', '15', '2', '3', '4', '5', '6', '7', '8', '9', 'pool-table', 'pool-table-hole', 'pool-table-side', 'solid', 'striped']


In [ ]:
# Find the index for 'pool-table-hole'
try:
    pocket_idx = master_classes.index('pool-table-hole')
    print(f"The index for 'pool-table-hole' is: {pocket_idx}")
    print(f"Suggested update: POCKET_CLASS_IDX = [{pocket_idx}]")
except ValueError:
    print(" 'pool-table-hole' not found in master_classes. Please check the list: ", master_classes)

The index for 'pool-table-hole' is: 17
Suggested update: POCKET_CLASS_IDX = [17]


In [ ]:
print(f'Total classes: {len(master_classes)}')
for i, name in enumerate(master_classes):
    print(f'Index {i}: {name}')

Total classes: 21
Index 0: 0
Index 1: 1
Index 2: 10
Index 3: 11
Index 4: 12
Index 5: 13
Index 6: 14
Index 7: 15
Index 8: 2
Index 9: 3
Index 10: 4
Index 11: 5
Index 12: 6
Index 13: 7
Index 14: 8
Index 15: 9
Index 16: pool-table
Index 17: pool-table-hole
Index 18: pool-table-side
Index 19: solid
Index 20: striped


In [ ]:
# Update Section 2/3 variables to use the combined dataset
USE_ROBOFLOW = False
DATASET_YAML = "/content/combined_dataset/data.yaml"

# Based on the combined class list, update POCKET_CLASS_IDX here
# From the previous check, 'pool-table-hole' is index 17
POCKET_CLASS_IDX = [17]

print(f'Ready to train with: {DATASET_YAML}')
print(f'Targeting pocket class indices: {POCKET_CLASS_IDX}')

In [ ]:
# Update Section 2/3 variables to use the combined dataset
USE_ROBOFLOW = False
DATASET_YAML = "/content/combined_dataset/data.yaml"

# Based on the combined class list, update POCKET_CLASS_IDX here if needed
# For example, if 'pocket' is index 2:
# POCKET_CLASS_IDX = [2]
print(f"Ready to train with: {DATASET_YAML}")

Ready to train with: /content/combined_dataset/data.yaml


## 2. Configuration

Fill these in, then run all cells.

## 3. Load Dataset & Inspect Classes

## 4. Train

In [1]:
from ultralytics import YOLO

# Configuration variables
EPOCHS = 50
IMGSZ = 640
BATCH = 32
DATASET_YAML = "/content/combined_dataset/data.yaml"

# Load a pretrained YOLOv8n model
model = YOLO("yolov8n.pt")

# Train the model on all 21 classes in the combined dataset
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=20,
    name="pocket_detector",
    project="/content/runs",
    # Augmentation: pockets appear at all angles and in varied lighting
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=45.0, flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
)

print("Best weights:", results.save_dir)

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
from ultralytics import YOLO
import os

# Configuration variables
EPOCHS = 50
IMGSZ = 640
BATCH = 32
DATASET_YAML = "/content/combined_dataset/data.yaml"

# Define a persistent save path on Google Drive
# This ensures weights are kept if the local runtime is deleted
PROJECT_DIR = "/content/drive/MyDrive/billiards_training"
os.makedirs(PROJECT_DIR, exist_ok=True)

# Load a pretrained YOLOv8n model
model = YOLO("yolov8n.pt")

# Train the model
# 'save=True' is default, weights will be in PROJECT_DIR/pocket_detector/weights
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=20,
    name="pocket_detector",
    project=PROJECT_DIR,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=45.0, flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
    save=True
)

print(f"Training complete. Weights saved to: {results.save_dir}")

## 5. Validate

In [ ]:
import os
best_weights = os.path.join(results.save_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)
metrics = trained_model.val(data=DATASET_YAML)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print()
if metrics.box.map50 > 0.85:
    print("✓ mAP50 > 0.85 — good to export")
else:
    print("⚠ mAP50 below 0.85 — consider more epochs or more data")

## 6. Export to TFLite FP16

In [ ]:
export_path = trained_model.export(
    format="tflite",
    imgsz=IMGSZ,
    half=True,
    int8=False,
    nms=True,
)
size_mb = os.path.getsize(export_path) / 1024 / 1024
print(f"TFLite: {export_path}  ({size_mb:.1f} MB)")

## 7. Smoke-test

Shows detections for the pocket class only (filtered by `POCKET_CLASS_IDX`).

In [ ]:
import numpy as np, cv2, glob, matplotlib.pyplot as plt, tensorflow as tf

interp = tf.lite.Interpreter(model_path=export_path)
interp.allocate_tensors()
inp_d = interp.get_input_details()
out_d = interp.get_output_details()
SZ    = int(inp_d[0]['shape'][1])
dtype = inp_d[0]['dtype']
print("Input:", inp_d[0]['shape'], dtype)

def infer(path):
    bgr = cv2.imread(path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    inp = np.expand_dims((cv2.resize(rgb, (SZ, SZ)) / 255.0).astype(dtype), 0)
    interp.set_tensor(inp_d[0]['index'], inp)
    interp.invoke()
    return rgb, interp.get_tensor(out_d[0]['index'])[0], w, h

yaml_dir = os.path.dirname(DATASET_YAML) + "/"
imgs = (glob.glob(yaml_dir + 'valid/images/*.jpg') +
        glob.glob(yaml_dir + 'valid/images/*.png'))[:4]

fig, axes = plt.subplots(1, max(len(imgs), 1), figsize=(5 * max(len(imgs), 1), 5))
if len(imgs) == 1: axes = [axes]

for ax, p in zip(axes, imgs):
    img, dets, w, h = infer(p)
    draw, n = img.copy(), 0
    for d in dets:
        # d = [x1, y1, x2, y2, conf, class_idx]  (with baked NMS)
        conf = float(d[4]) if len(d) >= 5 else 0
        cls  = int(d[5])   if len(d) >= 6 else 0
        if conf >= CONF_THRESHOLD and cls in POCKET_CLASS_IDX:
            x1,y1,x2,y2 = (int(d[0]*w/SZ), int(d[1]*h/SZ),
                            int(d[2]*w/SZ), int(d[3]*h/SZ))
            cv2.rectangle(draw, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(draw, f"{conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            n += 1
    ax.imshow(draw); ax.set_title(f"{n} pockets"); ax.axis('off')

plt.tight_layout(); plt.show()

## 8. Download

In [ ]:
import shutil
from google.colab import files
out = "/content/pocket_detector.tflite"
shutil.copy(export_path, out)
print(f"pocket_detector.tflite  ({os.path.getsize(out)/1024/1024:.1f} MB)")
files.download(out)

## 9. Android Integration (v1.4)

1. Copy `pocket_detector.tflite` → `app/src/main/assets/`

2. `app/build.gradle.kts`:
```kotlin
implementation("org.tensorflow:tensorflow-lite:2.16.1")
implementation("org.tensorflow:tensorflow-lite-support:0.4.4")
```

3. Implement `PocketDetector` — filter output by the pocket class index from this notebook:
```kotlin
class TflitePocketDetector(context: Context) : PocketDetector {
    // Class indices that correspond to pockets in your dataset
    private val pocketClassIndices = setOf(0)  // match POCKET_CLASS_IDX above

    private val interpreter = Interpreter(
        FileUtil.loadMappedFile(context, "pocket_detector.tflite")
    )

    override fun detect(yBytes: ByteArray, width: Int, height: Int): List<PointF>? {
        // 1. Resize yBytes to 640x640, replicate Y channel to RGB
        // 2. Run interpreter
        // 3. Parse output [x1,y1,x2,y2,conf,class_idx]
        //    return centres where conf >= 0.4 && class_idx in pocketClassIndices
    }
}
```

4. Pass to `TableScanAnalyzer` in `ProtractorScreen.kt`:
```kotlin
val detector = remember { TflitePocketDetector(context) }
val tableScanAnalyzer = remember(tableScanViewModel) {
    TableScanAnalyzer(
        tableScanViewModel::onFrame,
        tableScanViewModel::onFeltColorSampled,
        pocketDetector = detector,
    )
}
```